In [1]:
from debugpy.launcher import output
from dotenv import load_dotenv
from openai import OpenAI
import os

from openai.types.beta.threads import message

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("API_URL"),
)

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, built_index

documents = load_faq_data()
index = built_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
CONTEXT:
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
assistant.rag_pipeline("How do I run Olama locally?")

"You can serve a model locally with Ollama. Ollama allows you to run a model locally, so you don't need to make an external API call, and regional blocks don't apply. Most locally served models, including Ollama, expose an OpenAI-compatible endpoint, so the course code works with just a `base_url` change."

In [5]:
assistant.rag_pipeline("How do I run Ollama locally?")

'To run Ollama locally, follow these steps:\n\n1. Install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n   - **macOS**: Download the `.pkg` and install it.\n   - **Windows**: Download the `.msi` and install it.\n   - **Linux**: Run the following command in the terminal: \n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n2. Once installed, open a terminal and type:\n   ```bash\n   ollama run llama3\n   ```\n   This command will download the LLaMA 3 model (~4GB), start the model locally, and open a chat-like interface where you can type questions.\n3. To test the Ollama local server, run the following command:\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should receive a response similar to:\n   ```json\n   {"models": [...]}  \n   ```\n4. Then, install the Python client with:\n   ```bash\n   pip install ollama\n   ```\n5. You can now use Ollama in your Python scripts. Here is a 

In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [39]:
search_tool ={
            "type": "function",
            "name": "search",
            "description":"search in the FAQ database for entries match the given query",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query text to look up in the course FAQ."
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }




In [21]:

instructions="""You are a helpful assistant with access to tools.
    You must call tools using valid JSON ONLY.
    You're task is to Answer the QUESTION based on the CONTEXT from the FAQ database.
    """

In [22]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]


In [23]:
response = openai_client.responses.create(
    model="openai/gpt-oss-20b",
    input=messages,
    instructions=instructions,
    tools=[search_tool],
)

In [24]:
import json
call = response.output[1]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [25]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [26]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseReasoningItem(id='resp_01kwrqdrr3fqqvfeja0n70ndtt', summary=[], type='reasoning', content=[Content(text='We need to answer "I just discovered the course. Can I join it?" The user discovered the course, asks if they can join. We should refer to FAQ. Likely there is a FAQ entry about enrolling after discovery. Need to search.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"join after discovering course"}', call_id='fc_26a96ff0-bf38-4bad-9927-2caa01195b47', name='search', type='function_call', id='fc_26a96ff0-bf38-4bad-9927-2caa01195b47', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'fc_26a96ff0-bf38-4bad-9927-2caa01195b47',
  'output': '[\n  {\n    "id": "2d8b16c2a0",\n    "course": "mlops-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course - Can I still join

In [27]:
response = openai_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
    instructions=instructions,
    tools=[search_tool],
).output_text

In [28]:
response

'Yes, you can join the course after it has started. You can still register for the course and take part in it, but you might miss some of the homework submissions. To get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. The course materials will be available even after the course finishes, so you can follow it at your own pace.'

In [29]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [30]:
def make_call(call):
    args = json.loads(call.arguments)
    if call.name == "search":
        results = search(**args)
    result_json = json.dumps(results, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [31]:
question = "I just discovered the course. Can I join it?"
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
    tools=[search_tool],
)
messages.extend(response)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"course enrollment deadline"}
function_call: search {"query":"joining course after start date"}
function_call: search {"query":"late course registration"}


In [45]:
def agent_loop(instructions, question, model="llama-3.3-70b-versatile")->str:
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1
    last_answer = ''
    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model="llama-3.3-70b-versatile",
            input=messages,
            tools=[search_tool],
        )
        messages.extend(response.output)
        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it+=1
        if not has_function_calls:
            break
    return last_answer



In [47]:
answer = agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local setup instructions"}
function_call: search {"query":"running Olama on local machine"}
function_call: search {"query":"Olama development environment setup"}
iteration #2...
function_call: search {"query":"Olama local environment setup guide"}
iteration #3...
ASSISTANT:
It seems like the search results didn't provide a clear answer to your question. I'll try searching again with different keywords.


function_call: search {"query":"Olama setup instructions"}
iteration #4...
ASSISTANT:
Unfortunately, I couldn't find any specific instructions on running Olama locally. If you have any more details or context about Olama, I'd be happy to try and help you further. Are there any other areas you'd like to explore?


In [48]:
answer

"Unfortunately, I couldn't find any specific instructions on running Olama locally. If you have any more details or context about Olama, I'd be happy to try and help you further. Are there any other areas you'd like to explore?"

In [49]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"joining course deadline"}
function_call: search {"query":"course enrollment process"}
iteration #2...
function_call: search {"query":"joining the course now"}
function_call: search {"query":"late course enrollment"}
function_call: search {"query":"current course enrollment"}
function_call: search {"query":"enroll in current course"}
iteration #3...
ASSISTANT:
It appears that you can still join the course, but you might have missed some of the homework submissions. The course materials will be available after the course finishes, so you can follow it at your own pace. It's recommended to check the course repository and Slack channel for more information on how to get started.

Are there any other areas you'd like to explore or any specific questions you have about the course?


"It appears that you can still join the course, but you might have missed some of the homework submissions. The course materials will be available after the course finishes, so you can follow it at your own pace. It's recommended to check the course repository and Slack channel for more information on how to get started.\n\nAre there any other areas you'd like to explore or any specific questions you have about the course?"